# Train one kind (Colab, L4)

Fine-tunes one decision kind from the published Laya checkpoint that scores higher zero-shot on its validation split, at the reviewed commit, and scores validation before and after. No paid call is made here; the gate test is never read.

On the PC, after `laya/label.py` has labelled `train` and `validation` (and `calibration`, if temperatures are to be fitted):

    python laya/train.py pack useful

Upload `laya/data/useful/colab-useful.zip` to `MyDrive/laya/` and run this notebook on an L4 runtime. The checkpoint comes back to the same Drive folder.

The improvement loop (`laya/loop.py`) labels with paid host turns, so it runs on the PC, not here.

In [ ]:
import zipfile

from google.colab import drive

KIND = "useful"
DRIVE = "/content/drive/MyDrive/laya"

drive.mount("/content/drive")
zipfile.ZipFile(f"{DRIVE}/colab-{KIND}.zip").extractall("/content")
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q laya==0.4.6 transformers==5.19.0

In [ ]:
import sys

sys.path.insert(0, "/content/laya")   # this folder's modules by bare name; `import laya` stays the package

import label  # noqa: E402
import train  # noqa: E402

for split in train.SPLITS:
    rows = label.read(label.DATA / KIND / f"{split}.jsonl")
    print(split, len(rows), "states", sum(len(r["expected"]) for r in rows), "questions")

## Zero-shot

Both published checkpoints on validation; the better one is the base (plan stage 4).

In [ ]:
def accuracy(wrong, predicted):
    return 1 - len(wrong) / len(predicted)


zeroshot = {name: train.score(KIND, name) for name in train.PUBLISHED}
for name, (wrong, predicted) in zeroshot.items():
    print(f"{name}: {accuracy(wrong, predicted):.3f} on {len(predicted)} questions")
BASE = max(zeroshot, key=lambda n: accuracy(*zeroshot[n]))
BASE

## Fine-tune

fp16 with gradient checkpointing (Laya's CUDA defaults). A micro-batch of 4 with 16 accumulation steps keeps the notebook's effective batch of 64 inside the L4's 24 GB at `train.MAX_LEN`. `finetune` prints validation before and after; the checkpoint folder's `laya-run.json` records the run.

In [ ]:
checkpoint = train.train(KIND, ["train"], base=BASE, micro_batch=4, grad_accum=16)
wrong, predicted = train.score(KIND, checkpoint)
print(f"fine-tuned: {accuracy(wrong, predicted):.3f} on {len(predicted)} questions")

In [ ]:
import shutil

print(shutil.make_archive(f"{DRIVE}/{checkpoint.name}", "zip", checkpoint))